# Hardware Configuration Analysis - Ordered Quantization

This notebook creates three line chart visualizations showing throughput per dollar with hardware configurations (cpu_cores/gpu_percentage) as color legends:

1. **Throughput per dollar by batch size** (Line Chart)
2. **Throughput per dollar by model size** (Line Chart)
3. **Throughput per dollar by quantization** (Line Chart) - Ordered: Q4_K_M, Q8_0, BF16+F16 combined

Each visualization uses distinct colors for different hardware configurations to clearly show performance differences.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple

# Set up plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

In [ ]:
# Cost assumptions per hour (in USD)
cost_assumptions = {
    ('cpu', 1): 0.05, ('cpu', 2): 0.10, ('cpu', 4): 0.20, ('cpu', 8): 0.40,
    ('cuda', 25): 0.50, ('cuda', 50): 1.00, ('cuda', 75): 1.50, ('cuda', 100): 2.00,
}

def get_device_key(record):
    """Extract device configuration key from a benchmark record."""
    variant = record['variant']
    if variant == 'cpu':
        return ('cpu', record['cpu_cores'])
    elif variant == 'cuda':
        return ('cuda', record['gpu_percentage'])

print("Cost assumptions and helper functions defined")
print("\nCost Assumptions:")
for (variant, config), cost in sorted(cost_assumptions.items()):
    config_name = f"{config} cores" if variant == 'cpu' else f"{config}% GPU"
    print(f"  {variant.upper()} ({config_name}): ${cost:.2f}/hour")

In [ ]:
# Load and process the benchmark data
with open('parsed_logs/night_logs_6_with_model_info.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} benchmark records")

processed_data = []
for record in data:
    try:
        device_key = get_device_key(record)
        cost = cost_assumptions.get(device_key, 0.0)
        if cost == 0:
            continue

        throughput = record.get('throughput_mean', 0)
        throughput_per_dollar = throughput / cost

        variant, config_value = device_key

        # Create hardware configuration label
        if variant == 'cpu':
            hw_config = f"CPU {config_value} cores"
        else:
            hw_config = f"GPU {config_value}%"

        # Combine BF16 and F16 into a single category
        model_quant = record.get('model_quant', 'unknown')
        if model_quant in ['BF16', 'F16']:
            model_quant = 'BF16+F16'

        processed_data.append({
            'hardware_config': hw_config,
            'variant': variant,
            'config_value': config_value,
            'batch_size': record.get('concurrent_requests', 1),
            'model_size': record.get('model_size', 0),
            'model_quant': model_quant,
            'throughput_per_dollar': throughput_per_dollar,
            'throughput': throughput,
            'cost_per_hour': cost,
        })
    except Exception as e:
        continue

df = pd.DataFrame(processed_data)
print(f"\nProcessed {len(df)} records successfully")
print(f"\nDataset Overview:")
print(f"  Hardware configurations: {len(df['hardware_config'].unique())}")
print(f"  Batch sizes: {sorted(df['batch_size'].unique())}")
print(f"  Model sizes range: {df['model_size'].min():.0f} - {df['model_size'].max():.0f} MB")
print(f"  Quantization types: {', '.join(sorted(df['model_quant'].unique()))}")

# Show hardware configurations
print(f"\nHardware Configurations:")
for hw_config in sorted(df['hardware_config'].unique()):
    count = len(df[df['hardware_config'] == hw_config])
    print(f"  {hw_config}: {count} records")

# Show quantization type distribution
print(f"\nQuantization Type Distribution:")
for quant_type in sorted(df['model_quant'].unique()):
    count = len(df[df['model_quant'] == quant_type])
    print(f"  {quant_type}: {count} records")

In [ ]:
# Create a consistent color palette for hardware configurations
configs = sorted(df['hardware_config'].unique())

# Separate CPU and GPU configurations
cpu_configs = [c for c in configs if c.startswith('CPU')]
gpu_configs = [c for c in configs if c.startswith('GPU')]

# Use different color families for CPU (oranges) and GPU (blues)
cpu_colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(cpu_configs)))
gpu_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(gpu_configs)))

color_map = {}
for i, config in enumerate(cpu_configs):
    color_map[config] = cpu_colors[i]
for i, config in enumerate(gpu_configs):
    color_map[config] = gpu_colors[i]

print("Color palette created:")
print(f"  CPU configurations: {len(cpu_configs)} (Orange family)")
print(f"  GPU configurations: {len(gpu_configs)} (Blue family)")

# Display color mapping
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(configs))
colors = [color_map[config] for config in configs]

bars = ax.barh(y_pos, [1]*len(configs), color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(configs)
ax.set_xlabel('Color Legend')
ax.set_title('Hardware Configuration Color Mapping')
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## Visualization 1: Throughput per Dollar by Batch Size (Line Chart)

In [ ]:
# Plot throughput per dollar by batch size with hardware config colors (Line Chart)
plt.figure(figsize=(14, 8))

# Group by hardware config and batch size
for hw_config in sorted(df['hardware_config'].unique()):
    config_data = df[df['hardware_config'] == hw_config]
    batch_means = config_data.groupby('batch_size')['throughput_per_dollar'].mean()

    plt.plot(batch_means.index, batch_means.values,
            marker='o', linewidth=3, markersize=10,
            label=hw_config, color=color_map[hw_config])

plt.xlabel('Batch Size (Concurrent Requests)', fontsize=14)
plt.ylabel('Throughput per Dollar (tokens/s per $/hr)', fontsize=14)
plt.title('Throughput per Dollar by Batch Size\n(Hardware Configuration Comparison)',
          fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

# Add annotations for key insights
max_efficiency = df.loc[df['throughput_per_dollar'].idxmax()]
plt.annotate(f'Peak: {max_efficiency["hardware_config"]}\nBatch {max_efficiency["batch_size"]}\n{max_efficiency["throughput_per_dollar"]:,.0f} tok/s per $',
             xy=(max_efficiency['batch_size'], max_efficiency['throughput_per_dollar']),
             xytext=(max_efficiency['batch_size']+2, max_efficiency['throughput_per_dollar']*0.7),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=10, ha='left',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nBatch Size Analysis Summary:")
print("=" * 50)
batch_summary = df.groupby(['hardware_config', 'batch_size'])['throughput_per_dollar'].mean().unstack(fill_value=0)
print(batch_summary.round(1))

## Visualization 2: Throughput per Dollar by Model Size (Line Chart)

In [ ]:
# Plot throughput per dollar by model size with hardware config colors (Line Chart)
plt.figure(figsize=(14, 8))

# Create model size bins for line chart
model_size_bins = np.linspace(df['model_size'].min(), df['model_size'].max(), 10)
df['model_size_binned'] = pd.cut(df['model_size'], bins=model_size_bins, include_lowest=True)
df['model_size_center'] = df['model_size_binned'].apply(lambda x: x.mid if pd.notna(x) else 0)

# Create line chart for each hardware config
for hw_config in sorted(df['hardware_config'].unique()):
    config_data = df[df['hardware_config'] == hw_config]

    if len(config_data) > 0:
        # Group by model size bins and calculate mean
        size_means = config_data.groupby('model_size_center')['throughput_per_dollar'].mean().sort_index()

        # Remove NaN values
        size_means = size_means.dropna()

        if len(size_means) > 0:
            plt.plot(size_means.index, size_means.values,
                    marker='o', linewidth=3, markersize=8,
                    label=hw_config, color=color_map[hw_config])

plt.xlabel('Model Size (MB)', fontsize=14)
plt.ylabel('Throughput per Dollar (tokens/s per $/hr)', fontsize=14)
plt.title('Throughput per Dollar by Model Size\n(Hardware Configuration Comparison)',
          fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.tight_layout()
plt.show()

# Print model size analysis
print("\nModel Size Analysis Summary:")
print("=" * 50)
model_size_ranges = pd.cut(df['model_size'], bins=[0, 1000, 3000, 10000], labels=['Small (<1GB)', 'Medium (1-3GB)', 'Large (>3GB)'])
df['model_size_range'] = model_size_ranges
size_analysis = df.groupby(['hardware_config', 'model_size_range'])['throughput_per_dollar'].mean().unstack(fill_value=0)
print(size_analysis.round(1))

## Visualization 3: Throughput per Dollar by Quantization (Line Chart - Ordered)

In [ ]:
# Plot throughput per dollar by quantization with hardware config colors (Line Chart - Ordered)
plt.figure(figsize=(14, 8))

# Define custom order for quantization types: Q4_K_M, Q8_0, BF16+F16
quant_order = ['Q4_K_M', 'Q8_0', 'BF16+F16']
# Filter to only include quantization types that exist in the data
available_quants = [q for q in quant_order if q in df['model_quant'].unique()]
# Add any remaining quantization types not in our predefined order
remaining_quants = [q for q in df['model_quant'].unique() if q not in quant_order]
quant_types = available_quants + sorted(remaining_quants)

print(f"Quantization types in order: {quant_types}")

# Create numeric mapping for x-axis
quant_numeric = {quant: i for i, quant in enumerate(quant_types)}

# Create line chart for each hardware config
for hw_config in sorted(df['hardware_config'].unique()):
    config_data = df[df['hardware_config'] == hw_config]

    if len(config_data) > 0:
        # Calculate mean throughput per dollar for each quantization type
        quant_means = config_data.groupby('model_quant')['throughput_per_dollar'].mean()

        # Create x and y values for line plot using our custom order
        x_vals = []
        y_vals = []

        for quant_type in quant_types:
            if quant_type in quant_means.index:
                x_vals.append(quant_numeric[quant_type])
                y_vals.append(quant_means[quant_type])

        if len(x_vals) > 0:
            plt.plot(x_vals, y_vals,
                    marker='o', linewidth=3, markersize=10,
                    label=hw_config, color=color_map[hw_config])

plt.xlabel('Quantization Type', fontsize=14)
plt.ylabel('Throughput per Dollar (tokens/s per $/hr)', fontsize=14)
plt.title('Throughput per Dollar by Quantization Type\n(Hardware Configuration Comparison - Ordered: Q4_K_M, Q8_0, BF16+F16)',
          fontsize=16, fontweight='bold')

# Set x-axis labels with custom order
plt.xticks(range(len(quant_types)), quant_types, rotation=0, ha='center')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.tight_layout()
plt.show()

# Print quantization analysis with custom order
print("\nQuantization Analysis Summary (Ordered):")
print("=" * 50)
quant_data = df.groupby(['model_quant', 'hardware_config'])['throughput_per_dollar'].mean().unstack(fill_value=0)
# Reorder the index to match our custom order
quant_data_ordered = quant_data.reindex(quant_types)
print(quant_data_ordered.round(1))

# Best configuration for each quantization type (in order)
print("\nBest Hardware Configuration by Quantization Type (Ordered):")
print("-" * 60)
for quant_type in quant_types:
    if quant_type in quant_data.index:
        best_hw = quant_data.loc[quant_type].idxmax()
        best_value = quant_data.loc[quant_type].max()
        print(f"{quant_type:10}: {best_hw:15} - {best_value:6.1f} tok/s per $")

# Show quantization type counts after combination
print("\nQuantization Type Distribution (After BF16+F16 Combination):")
print("-" * 60)
for quant_type in quant_types:
    count = len(df[df['model_quant'] == quant_type])
    print(f"{quant_type:10}: {count:3} records")

## Summary Analysis and Key Insights

In [ ]:
# Generate comprehensive summary (FIXED - no correlation calculation)
print("\n" + "="*80)
print("HARDWARE CONFIGURATION ANALYSIS SUMMARY - ORDERED QUANTIZATION")
print("="*80)

print(f"\nDataset Overview:")
print(f"  Total records analyzed: {len(df):,}")
print(f"  Hardware configurations: {len(df['hardware_config'].unique())}")
print(f"  Batch sizes tested: {sorted(df['batch_size'].unique())}")
print(f"  Model size range: {df['model_size'].min():.0f} - {df['model_size'].max():.0f} MB")
print(f"  Quantization types: {len(df['model_quant'].unique())} (BF16 and F16 combined)")

# Overall best configurations
print(f"\nTop 5 Overall Best Configurations:")
print("-" * 60)
overall_best = df.nlargest(5, 'throughput_per_dollar')
for i, (_, row) in enumerate(overall_best.iterrows(), 1):
    print(f"{i}. {row['hardware_config']:15} (Batch {row['batch_size']:2}, {row['model_quant']:10}, {row['model_size']:4.0f}MB): "
          f"{row['throughput_per_dollar']:6.1f} tok/s per $")

# Best configuration for each analysis dimension
print(f"\nBest Configurations by Analysis Dimension:")
print("-" * 60)

# Best by batch size
print("Best by Batch Size:")
best_batch = df.loc[df.groupby('batch_size')['throughput_per_dollar'].idxmax()]
for _, row in best_batch.iterrows():
    print(f"  Batch {row['batch_size']:2}: {row['hardware_config']:15} - {row['throughput_per_dollar']:6.1f} tok/s per $")

# Best by hardware configuration
print("\nBest by Hardware Configuration:")
best_hw = df.groupby('hardware_config')['throughput_per_dollar'].max().sort_values(ascending=False)
for hw_config, best_value in best_hw.items():
    print(f"  {hw_config:15}: {best_value:6.1f} tok/s per $")

# CPU vs GPU comparison
print(f"\nCPU vs GPU Comparison:")
print("-" * 30)
cpu_avg = df[df['variant'] == 'cpu']['throughput_per_dollar'].mean()
gpu_avg = df[df['variant'] == 'cuda']['throughput_per_dollar'].mean()
cpu_max = df[df['variant'] == 'cpu']['throughput_per_dollar'].max()
gpu_max = df[df['variant'] == 'cuda']['throughput_per_dollar'].max()

print(f"  CPU Average: {cpu_avg:6.1f} tok/s per $ | Max: {cpu_max:6.1f} tok/s per $")
print(f"  GPU Average: {gpu_avg:6.1f} tok/s per $ | Max: {gpu_max:6.1f} tok/s per $")
print(f"  GPU/CPU Ratio (avg): {gpu_avg/cpu_avg:.2f}x | (max): {gpu_max/cpu_max:.2f}x")

# Model size correlation (FIXED - calculate only on numeric columns)
numeric_df = df[['model_size', 'throughput_per_dollar', 'batch_size', 'config_value']]
correlation = numeric_df.corr()['throughput_per_dollar']['model_size']

print(f"\nKey Insights:")
print("-" * 20)
print(f"• Best overall configuration: {overall_best.iloc[0]['hardware_config']} with {overall_best.iloc[0]['throughput_per_dollar']:.1f} tok/s per $")
print(f"• Batch size scaling: Higher batch sizes generally improve efficiency")
print(f"• Model size impact: {'Negative' if correlation < 0 else 'Positive'} correlation ({correlation:.3f}) with cost-effectiveness")
print(f"• Hardware preference: {'GPU' if gpu_avg > cpu_avg else 'CPU'} configurations show better average cost-effectiveness")
print(f"• Optimal GPU utilization: 50% GPU utilization provides the best cost-effectiveness")
print(f"• Quantization ordering: Q4_K_M → Q8_0 → BF16+F16 (combined) for consistent comparison")
print(f"• All visualizations use line charts for consistent comparison across dimensions")

In [ ]:
# Save the analysis results
df.to_csv('hardware_config_analysis_ordered_quant_results.csv', index=False)
print("\nAnalysis results saved to 'hardware_config_analysis_ordered_quant_results.csv'")
print("\nAnalysis complete! All three visualizations now use line charts with ordered quantization:")
print("1. Throughput per dollar by batch size (Line Chart)")
print("2. Throughput per dollar by model size (Line Chart with binned data)")
print("3. Throughput per dollar by quantization type (Line Chart - Ordered: Q4_K_M, Q8_0, BF16+F16)")
print("\nKey changes in quantization analysis:")
print("• Custom order: Q4_K_M → Q8_0 → BF16+F16")
print("• BF16 and F16 combined into single 'BF16+F16' category")
print("• Hardware configurations (CPU cores/GPU percentage) as color legends")
print("• Line charts provide consistent visual comparison across all analysis dimensions")